# Does more training data help? Learning curve over the number of training reviews (Qwen3.5-0.8B)

The 0.8B run ([train_qwen35_0.8b.ipynb](train_qwen35_0.8b.ipynb)) peaks at MAP 21.4 (abstract level) after 3 epochs
and then memorises the train set (train loss 0.0001 after 10 epochs), while Qwen3.5-4B reaches about 60 on the
same 92 SYNERGY+ train reviews. The train set has only **92 distinct criteria texts**. This notebook tests whether
more reviews would help, by training on fewer: if the score still rises clearly from 46 to 92 reviews, more reviews
will probably help further; if it is flat, more data of this kind is not the bottleneck.

| | |
|---|---|
| Model | `unsloth/Qwen3.5-0.8B` (pinned), LoRA r 16 + new decision head, lr 2e-4 (head 1e-4), batch 64 × 1, seed 3407: as in the 0.8B notebook |
| Train data | the balanced train requests (1 exclude per include per review); **only the number of reviews changes**: k = 12, 23, 46 random subsets, and 92 (all) |
| k = 92 | a fresh run with the same script. The earlier 0.8B run was meant to be reused here, but it stalled at the start (train loss 0.69 for 200+ steps; best MAP 21.4, while k = 46 reached 52.9), so it is only a reference. Every run keeps its training settings and learning-rate schedule |
| Subsets | [scripts/subset_train.py](../../scripts/subset_train.py): stratified by domain (shares as in the 92) and by review size; all requests of a chosen review are kept |
| Training length | **1,266 steps for every run** (= 3 epochs of the full set, where the 0.8B run peaked). Cosine schedule over 4,220 steps, no warmup (the 10-epoch schedule of the 0.8B run), stopped at 1,266, so the learning rate falls from 2e-4 to 1.59e-4 in every run. Equal compute for every k; small subsets are repeated more often (k = 12: about 20 epochs), which is what having less data means |
| Validation | the same 27,870 records of the 17 CLEF TAR 2019 Intervention training reviews for every run, scored at steps 50, 100, 200, 422, 844, 1266 (the last three are the 0.8B run's epoch evaluations; the early ones because small subsets overfit early) |
| Reported | best step by abstract-level MAP (same selection for every run, slightly optimistic for all) and the last step; abstract-level and content-level labels |
| Runs | [scripts/train_decision_model.py](../../scripts/train_decision_model.py), one process per run, physical GPU 1 only; about 1.5–2 h per run |

**Runs in stages** (stop as soon as the answer is clear):

1. k = 46 seed 1 and k = 92: gain or flat?
2. k = 46 seed 2: only if the stage-1 gain is near the 3-MAP threshold, to see how much of it is the choice of subset.
3. k = 12 and 23 (seed 1): the curve shape, only if there is a gain (how much would more reviews give?).

**Decision rule.** Gain from 46 to 92 reviews (per-review bootstrap CI): at least about 3 MAP with a CI above 0 →
collecting more reviews is worth it. Flat → more data of this kind will not fix 0.8B; look at capacity or training
instead. The CI covers the val reviews only, not the choice of train subset; with one subset per k that part is
unknown, which is what stage 2 is for.

In [ ]:
import json, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
from screening import metrics

SOURCE = "review-criteria__record-title+abstract__label-abstract_else_final__train-neg1"
MODEL = "unsloth/Qwen3.5-0.8B"
MAX_STEPS = 1266          # training stops here (3 epochs of the full set)
SCHEDULE_STEPS = 4220     # cosine schedule length, as in the 0.8B run (10 epochs)
EVAL_STEPS = [50, 100, 200, 422, 844, 1266]
PYTHON = sys.executable

PLAN = [(46, 1),             # stage 1 (with k = 92): gain or flat?
        (46, 2),             # stage 2: only if the stage-1 gain is near the 3-MAP threshold
        (12, 1), (23, 1)]    # stage 3: curve shape, only if there is a gain
PLAN = [(46, 1), (92, 1)] + PLAN[1:]   # k = 92 is a fresh run: the old 0.8B run stalled at the start (see above)
RUN_NOW = PLAN[:2]           # extend after looking at the result of each stage
CONFIGS = PLAN               # everything reported (only finished runs are shown)

REQ = ROOT / "data" / "processed" / "training" / "requests"
JEV_SCORES = ROOT / "results" / f"requests__{SOURCE.removesuffix('__train-neg1')}__val" / "scores.jsonl"
OLD_08B = ROOT / "runs" / f"Qwen3.5-0.8B__{SOURCE}"   # old run, stalled at the start: reference only
COMPARE = {"Qwen3.5-4B (best epoch)": ROOT / "runs" / f"Qwen3.5-4B__{SOURCE}",
           "Qwen3.5-0.8B old run, stalled start (best epoch)": OLD_08B}

def name(k, s): return f"{SOURCE}__k{k}-s{s}"
def run_dir(k, s):
    return ROOT / "runs" / f"{MODEL.split('/')[-1]}__{name(k, s)}__steps{MAX_STEPS}__sched{SCHEDULE_STEPS}"
def finished(k, s):
    path = run_dir(k, s) / "val_metrics.jsonl"
    return path.exists() and any(json.loads(l)["step"] == MAX_STEPS for l in path.open())

print(len(RUN_NOW), "runs to train now:", RUN_NOW)

## Subsets

In [ ]:
rows = []
for k, s in CONFIGS:
    if not (REQ / name(k, s) / "config.json").exists():
        subprocess.run([PYTHON, str(ROOT / "scripts" / "subset_train.py"), "--source", SOURCE,
                        "--reviews", str(k), "--seed", str(s)], check=True)
    cfg = json.loads((REQ / name(k, s) / "config.json").read_text())
    rows.append({"k": k, "seed": s, **cfg["subset_train"],
                 "epochs at the cap": round(MAX_STEPS * 64 / cfg["subset_train"]["requests"], 1)})
subsets = pd.DataFrame(rows).set_index(["k", "seed"]).sort_index()
subsets

## Training (in the background)

The runs in `RUN_NOW` are started one after the other in a detached tmux session (`review-count-curve`) on this
machine, so they keep running when VS Code or the notebook kernel is closed. The cell returns at once; the notebook
only reads results, so it can be closed and reopened at any time. Watch with the status cell below, or in a terminal:
`tmux attach -t review-count-curve` (leave with Ctrl-b d).

Each run is a separate process (GPU memory is freed between runs). Finished runs are skipped; if a run fails, the
session stops. A run that stopped half-way must be moved away before starting again (the script refuses to append to
existing results). Output of each run: `<run>/train.out`. GPU 1 must be free: shut down idle notebook kernels that
hold it.

In [ ]:
import shlex
SESSION = "review-count-curve"

def tmux_running():
    return subprocess.run(["tmux", "has-session", "-t", SESSION], capture_output=True).returncode == 0

todo = [(k, s) for k, s in RUN_NOW if not finished(k, s)]
half = [(k, s) for k, s in todo if (run_dir(k, s) / "val_metrics.jsonl").exists()]
assert not half, f"stopped half-way, move these run folders away first: {[str(run_dir(k, s)) for k, s in half]}"
if tmux_running():
    print(f"tmux session {SESSION!r} is still running: not starting another")
elif not todo:
    print("all runs in RUN_NOW are finished")
else:
    steps = []
    for k, s in todo:
        run_dir(k, s).mkdir(parents=True, exist_ok=True)
        cmd = [PYTHON, str(ROOT / "scripts" / "train_decision_model.py"), "--requests", name(k, s), "--model", MODEL,
               "--max-steps", str(MAX_STEPS), "--schedule-steps", str(SCHEDULE_STEPS), "--batch", "64",
               "--eval-steps", ",".join(map(str, EVAL_STEPS))]
        out = shlex.quote(str(run_dir(k, s) / "train.out"))
        steps.append(f"echo \"$(date '+%F %T') k={k} seed={s}: start\"; {shlex.join(cmd)} > {out} 2>&1"
                     f" && echo \"$(date '+%F %T') k={k} seed={s}: done\""
                     f" || {{ echo \"$(date '+%F %T') k={k} seed={s}: FAILED, see {out}\"; exit 1; }}")
    script = " ; ".join(steps) + " ; echo all done"
    subprocess.run(["tmux", "new-session", "-d", "-s", SESSION, "-c", str(ROOT),
                    f"bash -c {shlex.quote(script)} 2>&1 | tee -a {shlex.quote(str(ROOT / 'runs' / (SESSION + '.log')))}; "
                    "sleep 86400"], check=True)   # keeps the window open a day after the end, for a look
    print(f"started in tmux session {SESSION!r}:", todo)

In [ ]:
# Status: run this any time (also after reopening the notebook)
print("tmux session running:", tmux_running())
log = ROOT / "runs" / f"{SESSION}.log"
if log.exists():
    print(log.read_text()[-1000:])
for k, s in PLAN:
    path = run_dir(k, s) / "val_metrics.jsonl"
    evals = [json.loads(l) for l in path.open()] if path.exists() else []
    last = f"last eval: step {evals[-1]['step']}, MAP {evals[-1]['map_abstract']}" if evals else "no eval yet"
    out = run_dir(k, s) / "train.out"
    tail = out.read_text(errors="replace").replace("\r", "\n").strip().splitlines()[-1][:150] if out.exists() and out.stat().st_size else ""
    print(f"k={k} seed={s}: {'finished' if finished(k, s) else last} | {tail}")

In [ ]:
# Progress plot: val MAP and WSS@95% per evaluation step for every run so far (also while a run is training);
# reference lines: Qwen3.5-4B and the old 0.8B run (best epoch, dashed), zero-shot Jev (dotted)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for (k, s), color in zip(PLAN, plt.cm.tab10.colors):
    path = run_dir(k, s) / "val_metrics.jsonl"
    if not path.exists():
        continue
    h = pd.read_json(path, lines=True).drop_duplicates("step", keep="last").set_index("step")
    for ax, m in zip(axes.flat, ("map_abstract", "map_final", "wss95_abstract", "wss95_final")):
        ax.plot(h.index, h[m], "o-", color=color, label=f"k={k} seed={s}" + ("" if finished(k, s) else " (running)"))
for ref_label, run in COMPARE.items():   # best epoch of the reference runs, as dashed lines
    if (run / "val_metrics.jsonl").exists():
        ref = pd.read_json(run / "val_metrics.jsonl", lines=True).query("step > 0")
        ref = ref.loc[ref["map_abstract"].idxmax()]
        for ax, m in zip(axes.flat, ("map_abstract", "map_final", "wss95_abstract", "wss95_final")):
            ax.axhline(ref[m], ls="--", lw=1, color="grey" if "4B" in ref_label else "lightgrey")
            ax.text(MAX_STEPS, ref[m], ref_label.split(" (")[0], ha="right", va="bottom", fontsize=7, color="grey")
# Zero-shot Jev on the same val records (ranked as in FINDINGS: ties broken by BM25, then record id)
jev_df = pd.read_json(JEV_SCORES, lines=True)
jev_ref = {}
for level in ("abstract", "final"):
    per = [m for _, g in jev_df.groupby("review_id")
           if (m := metrics(g.sort_values(["jev", "bm25", "record_id"], ascending=[False, False, True])[f"label_{level}"].astype(int)))]
    mean = pd.DataFrame(per).mean() * 100
    jev_ref[f"map_{level}"], jev_ref[f"wss95_{level}"] = mean["MAP"], mean["WSS@95%"]
for ax, m in zip(axes.flat, ("map_abstract", "map_final", "wss95_abstract", "wss95_final")):
    ax.axhline(jev_ref[m], ls=":", lw=1.5, color="tab:red")
    ax.text(EVAL_STEPS[0], jev_ref[m], "Jev zero-shot", ha="left", va="bottom", fontsize=7, color="tab:red")
for ax, m in zip(axes.flat, ("MAP, abstract-level labels", "MAP, content-level labels",
                             "WSS@95%, abstract-level labels", "WSS@95%, content-level labels")):
    ax.set_title(m); ax.set_xscale("log"); ax.set_xticks(EVAL_STEPS, EVAL_STEPS); ax.grid(alpha=0.3)
for ax in axes[1]:
    ax.set_xlabel("training step (log scale)")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Qwen3.5-0.8B: val (17 CLEF TAR 2019 reviews) by training step and number of train reviews k")
fig.tight_layout()

## Result: learning curve

Macro MAP / WSS@95% over the val reviews. Per run: best step by abstract-level MAP (steps ≤ 1266), and the last step
(1266). Per k: mean and range over the subsets that were run.

In [ ]:
def history(k, s):
    h = pd.read_json(run_dir(k, s) / "val_metrics.jsonl", lines=True).drop_duplicates("step", keep="last")
    h = h[(h["step"] > 0) & (h["step"] <= MAX_STEPS)].set_index("step")
    h["epoch"] = (h.index * 64 / subsets.loc[(k, s), "requests"]).round(2)   # the old run did not log it
    return h

runs = []
for k, s in CONFIGS:
    if not finished(k, s):
        continue
    h = history(k, s)
    best = h["map_abstract"].idxmax()
    runs.append({"k": k, "seed": s, "best step": best, "epoch at best": h.loc[best, "epoch"],
                 **{f"{m} best": h.loc[best, m] for m in ("map_abstract", "wss95_abstract", "map_final", "wss95_final")},
                 **{f"{m} last": h.loc[MAX_STEPS, m] for m in ("map_abstract", "wss95_abstract", "map_final", "wss95_final")}})
runs = pd.DataFrame(runs).set_index(["k", "seed"]).sort_index()
display(runs)

cols = [c for c in runs.columns if c.endswith(("best", "last")) and c != "best step"]
curve = runs[cols].groupby("k").agg(["mean", "min", "max"]).round(1)
curve

In [ ]:
# Reference point on the same val records
def best_epoch(run):
    h = pd.read_json(run / "val_metrics.jsonl", lines=True).drop_duplicates("step", keep="last").set_index("step")
    h = h.drop(0, errors="ignore")
    return h.loc[h["map_abstract"].idxmax(), ["map_abstract", "wss95_abstract", "map_final", "wss95_final"]]

refs = {label: best_epoch(run) for label, run in COMPARE.items() if (run / "val_metrics.jsonl").exists()}
pd.DataFrame(refs).T

In [ ]:
# Abstract-level MAP per evaluation step: where each run peaks, and how fast it overfits
steps = pd.concat({(k, s): history(k, s)["map_abstract"] for k, s in CONFIGS if finished(k, s)}, axis=1)
steps.T.sort_index().round(1)

## Per review: is the gain from doubling the reviews real?

Per val review, MAP at each run's best step, averaged over the subsets of a k. Differences between neighbouring k
that were run: mean over reviews, 95% bootstrap CI over reviews, reviews better / worse. The CI covers the val
reviews only, not the choice of train subsets; the range over subsets above shows that part.

In [ ]:
def per_review(df, col, level, tie=None):
    rows = {}
    for rid, g in df.groupby("review_id"):
        keys = [col] + ([tie] if tie else []) + ["record_id"]
        g = g.sort_values(keys, ascending=[False] * (len(keys) - 1) + [True])
        m = metrics(g[f"label_{level}"].to_numpy().astype(int))
        if m:
            rows[rid] = m
    return pd.DataFrame(rows).T * 100

rng = np.random.default_rng(0)
def diff(a, b):
    d = (a - b).dropna().to_numpy()
    lo, hi = np.percentile(rng.choice(d, (5000, len(d))).mean(1), [2.5, 97.5])
    return f"{d.mean():+.1f} [{lo:+.1f}, {hi:+.1f}] {(d > 0).sum()}/{(d < 0).sum()}"

out = []
jev = pd.read_json(JEV_SCORES, lines=True)
for level in ("abstract", "final"):
    by_k = {}
    for k in sorted({k for k, _ in CONFIGS}):
        per_run = [per_review(pd.read_json(run_dir(k, s) / "val_scores" / f"step-{runs.loc[(k, s), 'best step']:06d}.jsonl",
                                           lines=True), "p", level)
                   for kk, s in CONFIGS if kk == k and (k, s) in runs.index]
        if per_run:
            by_k[k] = pd.concat(per_run).groupby(level=0).mean()
    J = per_review(jev, "jev", level, "bm25")
    ks = sorted(by_k)
    for a, b in zip(ks, ks[1:]):
        out.append({"level": level, "comparison": f"k {b} − k {a}",
                    "ΔMAP": diff(by_k[b]["MAP"], by_k[a]["MAP"]), "ΔWSS@95%": diff(by_k[b]["WSS@95%"], by_k[a]["WSS@95%"])})
    for k in ks:
        out.append({"level": level, "comparison": f"k {k} − Jev",
                    "ΔMAP": diff(by_k[k]["MAP"], J["MAP"]), "ΔWSS@95%": diff(by_k[k]["WSS@95%"], J["WSS@95%"])})
pd.DataFrame(out)

## Reading the result

- **MAP still rising from 46 to 92** (CI above 0, gain of at least about 3): more reviews should help. Run stage 3
  (k = 12, 23) to see the step per doubling, which estimates what another 92 reviews would give. If the gain is near
  3, run stage 2 first: a second k = 46 subset shows whether the gain depends on which reviews were picked.
- **Flat from 46 on**: the 0.8B model does not get better from more reviews of this kind. Next: Step 0 of the plan
  (does the model use the criteria at all?) and the 4B curve at small k (is 4B already saturated?).
- **Best step moving later with larger k** is expected (more data overfits later). k = 92 peaks at 1266, the cap;
  it might have risen a little further, but the old run's next evaluation (step 1688) was lower.